In [28]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
load_dotenv()

True

In [29]:
model=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.getenv("GEMINI_API_KEY_2")
)

In [30]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str 
    eval_score:float

In [31]:
def createOutline(state:BlogState) -> BlogState:
    title=state['title']
    prompt=f"Genereate a detailes outline for a blog on the topic {title}"
    ans=model.invoke(prompt).content[0]['text']
    state['outline']=ans
    return state

In [32]:

def createBlog(state:BlogState) -> BlogState:
    
    title=state['title']
    outline=state['outline']
    
    prompt=f"Write a detailed blog on the title-{title} using the following outline \n{outline}"
    content=model.invoke(prompt).content[0]['text']
    state['content']=content
    return state

In [33]:
def evaluateResult(state:BlogState)->BlogState:
    outline=state['outline']
    blog=state['content']
    
    prompt=f"Based on this outline:{outline} can you evaluate this blog content: {blog} and return a integer score out of 10 rating the blog"
    eval_ans=model.invoke(prompt).content[0]['text']
    state['eval_score']=eval_ans
    return state

In [34]:
graph=StateGraph(BlogState)

graph.add_node('create_outline',createOutline)
graph.add_node('create_blog',createBlog)
graph.add_node('evaluate',evaluateResult)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','evaluate')
graph.add_edge('evaluate',END)

workflow=graph.compile()


In [35]:
initial_state= {'title':"Rise of AI in Nepal"}
final_state=workflow.invoke(initial_state)
print(final_state)

{'title': 'Rise of AI in Nepal', 'outline': 'Here is a detailed, SEO-friendly outline for a comprehensive blog post on the **"Rise of AI in Nepal."** \n\n---\n\n# Blog Title Options:\n*   *From the Himalayas to Algorithms: The Rise of AI in Nepal*\n*   *Artificial Intelligence in Nepal: Current Landscape, Challenges, and Future Prospects*\n*   *Decoding Nepal’s AI Revolution: How the Mountain Nation is Embracing the Future*\n\n---\n\n## **I. Introduction**\n*   **The Hook:** Briefly contrast Nepal’s traditional image (mountains, rich heritage, trekking) with its modern reality (a burgeoning tech-savvy youth and growing digital infrastructure).\n*   **Defining the Trend:** Introduce the rapid adoption and discussion around Artificial Intelligence (AI) in Nepal over the last few years.\n*   **Thesis Statement:** While still in its developmental stages, AI in Nepal is no longer just a sci-fi concept—it is actively transforming local businesses, education, governance, and social impact ini

In [36]:
final_state['content']

"# From the Himalayas to Algorithms: The Rise of AI in Nepal\n\nWhen the world thinks of Nepal, the imagery that instantly comes to mind is timeless: the snow-capped majesty of Mount Everest, fluttering prayer flags in the wind, ancient pagoda temples, and the warm hospitality of mountain villagers. \n\nYet, beneath this rich cultural and geographical heritage, a quiet revolution is taking place. Walk down the bustling streets of Kathmandu or the tech hubs of Lalitpur, and you will witness a different Nepal—one defined by a burgeoning, tech-savvy youth demographic, bustling co-working spaces, and a rapidly growing digital infrastructure.\n\nIn this shifting landscape, Artificial Intelligence (AI) has emerged as a major buzzword. While still in its developmental and exploratory stages, AI in Nepal is no longer just a sci-fi concept confined to Hollywood movies. It is actively transforming local businesses, modernizing education, reshaping governance, and driving social impact initiative

In [37]:
final_state['eval_score']

'**Score: 9.5 / 10**\n\n### **Evaluation & Feedback:**\n\n*   **Adherence to Outline (10/10):** The blog strictly follows the provided outline, covering every single section, subheading, and key point logically and seamlessly. \n*   **Tone & Style (10/10):** The tone is engaging, professional, optimistic, and culturally grounded. It brilliantly contrasts Nepal\'s traditional imagery with its modern technological ambitions, exactly as requested in the hook.\n*   **Flow & Readability (9.5/10):** The transitions between paragraphs are smooth. The use of bold text, bullet points, and clear section headers makes the article highly scannable and SEO-friendly. \n*   **Formatting Critique:** The only minor deduction is that the section headers still retain the outline\'s numbering (e.g., "## II. The Current Landscape..."), which should ideally be removed for a clean, published blog post format. \n\nOverall, this is a phenomenal, well-researched, and compelling piece of content ready for public